# Synthetic Floor 7-Stage — GPU Blender Renderer (Colab)

End-to-end notebook that:
1. Installs Blender 4.2 LTS portable on a Colab GPU runtime (T4 / A100).
2. Clones the MyCon repo and prepares the BIM + GLB + sidecar JSON for every stage.
3. Calls the GPU renderer (`blender_gpu_renderer.py`) per stage via Cycles + OptiX/CUDA + OpenImageDenoise.
4. Encodes per-stage MP4s and previews a few frames inline.

**Runtime:** open *Runtime › Change runtime type* and pick a **T4** or **A100** GPU before running cell 1.

## 1. Verify GPU + clone the repo

In [ ]:
import subprocess, os
print(subprocess.check_output(['nvidia-smi', '-L']).decode())
if not os.path.isdir('/content/MyCon'):
    !git clone --depth 1 https://github.com/sayyedalimrj/MyCon.git /content/MyCon
%cd /content/MyCon/repository

## 2. Install Blender 4.2 LTS (portable build)

In [ ]:
!bash examples/synthetic_floor_7stage/scripts/setup_colab_blender.sh

## 3. Install Python deps the host orchestrator uses

`bpy` is **not** needed on the host — those imports happen inside Blender. The host only needs the same lightweight stack the CPU example uses.

In [ ]:
!pip install -q numpy Pillow PyYAML trimesh imageio imageio-ffmpeg ifcopenshell scipy

## 4. Quick smoke test — stage 7 only, 480x270, 30 frames, 32 samples

Should finish in roughly **30–60 seconds** on a T4. Confirms the whole pipeline (GPU detection, mesh import, materials, sky, camera, compositor) is wired correctly.

In [ ]:
!PYTHONPATH=examples/synthetic_floor_7stage/src \
    python3 examples/synthetic_floor_7stage/scripts/run_blender_gpu.py \
        --blender /content/blender/blender \
        --stages 7 --quick

## 5. Preview a frame and the depth map

In [ ]:
from PIL import Image
import numpy as np
from pathlib import Path

rgb_dir = Path('examples/synthetic_floor_7stage/output/blender_renders/stage_07/rgb')
frames = sorted(rgb_dir.glob('frame_*.png'))
print(f'{len(frames)} RGB frames in {rgb_dir}')
Image.open(frames[len(frames)//2]).resize((640, 360))

In [ ]:
import imageio.v3 as iio
import matplotlib.pyplot as plt

depth_dir = Path('examples/synthetic_floor_7stage/output/blender_renders/stage_07/depth')
exrs = sorted(depth_dir.glob('frame_*.exr'))
if exrs:
    arr = iio.imread(exrs[len(exrs)//2])
    arr = np.array(arr)
    if arr.ndim == 3:
        arr = arr[..., 0]
    finite = arr[np.isfinite(arr) & (arr < 1e6)]
    if finite.size:
        print(f'depth range: {finite.min():.2f}\u2013{finite.max():.2f} m')
        plt.imshow(np.where(arr < 1e6, arr, np.nan), cmap='turbo')
        plt.colorbar(label='m')
        plt.title('Depth (m)')
        plt.axis('off')
        plt.show()

## 6. Full 7-stage render at 1280×720 — `~10–20 min on T4, ~5 min on A100`

In [ ]:
!PYTHONPATH=examples/synthetic_floor_7stage/src \
    python3 examples/synthetic_floor_7stage/scripts/run_blender_gpu.py \
        --blender /content/blender/blender \
        --resolution 1280 720 \
        --samples 128 \
        --device OPTIX

## 7. Inspect the manifest

In [ ]:
import json
manifest = json.load(open('examples/synthetic_floor_7stage/output/manifests/manifest_blender_gpu.json'))
print(json.dumps({
    'project': manifest['project'],
    'video': manifest['video'],
    'stage_files_keys': list(manifest['stage_files'].keys()),
}, indent=2))